# 03. Дообучение e5-large (для v8)

Тот же ноутбук, что `03_embeddings.ipynb`, с настройкой `EMB_VERSION = "large"`: дообучаем `intfloat/multilingual-e5-large` (560 млн параметров, вектор 1024) по схеме второго раунда v6, то есть трудные негативы без объявлений той же микрокатегории. Результат: артефакт `artifacts/embeddings_large`.

Зачем. Сама по себе large не лучше e5-base после двух раундов (Recall@100 0,422 против 0,420; ранкер на ней дал на валидации 0,919 против 0,923 у v6). Но это другая модель с другими ошибками, и в v8 средняя близость двух моделей подняла LB с 0,8628 до 0,8736.

Отличия от v6: одна эпоха на 150 тыс. парах (большая модель учится примерно в 2,5 раза медленнее), матрица эмбеддингов слов заморожена, иначе на 20 ГБ помещается батч только ~64. Время выполнения: ~3 ч на A100 20 ГБ.

## Настройки

Единственная ячейка, которую может понадобиться поправить; `None` означает «определить автоматически». Проверка кода без GPU: `SMOKE_TEST = True` (крошечная модель, ничего не скачивается).

In [1]:
# ── Пути (None: автоматически) ──────────────────────────────────────────────────────────
REPO_DIR = None       # папка репозитория, если ноутбук открыт не из его папки notebooks/
DATA_DIR = None       # папка с train.parquet и benchmark_*.parquet; по умолчанию <репозиторий>/data
WORK_DIR = None       # куда сохранить артефакт; по умолчанию <репозиторий>/artifacts
HF_HOME = None        # кэш моделей Hugging Face (~1,1 ГБ на модель); по умолчанию ~/.cache/huggingface
HF_ENDPOINT = None    # зеркало, если huggingface.co недоступен, например "https://hf-mirror.com"

# ── Режим ─────────────────────────────────────────────────────────────────────────────────
DRY_RUN = False       # True: все шаги на маленьких подвыборках, ~10 минут (проверка перед полным запуском)
SMOKE_TEST = False    # True: крошечная модель со случайными весами, ничего не скачивается (проверка кода)

# ── Версия артефакта ──────────────────────────────────────────────────────────────────────
EMB_VERSION = "large" # "v6": второй раунд от модели v4, негативы без той же микрокатегории → artifacts/embeddings_v6
                      # "v4": первый раунд с нуля, как в теге v4 → artifacts/embeddings
                      # "large": multilingual-e5-large, один раунд с очищенными негативами → artifacts/embeddings_large

# ── Обучение (None / 0: значения из src/config.py) ──────────────────────────────────────
MODEL_CANDIDATES = None   # список имён на Hugging Face или путей к скачанным папкам;
                          # ["intfloat/multilingual-e5-base"]: без сравнения моделей (экономит ~15 минут)
BATCH_SIZE = 0            # 0: подобрать под память GPU
OVERWRITE = False         # True: разрешить перезапись уже готового артефакта той же версии
TRAIN_PAIRS = None        # пар для дообучения; None: из конфига версии (v4: 400 000, v6: все свободные, large: 150 000)

## 0. Окружение

Подключаем код из `src/` и ставим недостающие пакеты. Число потоков BLAS фиксируется до импорта numpy.

In [ ]:
import base64, importlib, os, subprocess, sys
from pathlib import Path

for _name in ("REPO_DIR", "DATA_DIR", "WORK_DIR", "OUTPUT_DIR", "EMB_DIR", "HF_HOME", "HF_ENDPOINT"):
    if globals().get(_name):
        os.environ[_name] = str(globals()[_name])
for _name in ("DRY_RUN", "SMOKE_TEST"):
    if globals().get(_name):
        os.environ[_name] = "1"

N_THREADS = 4
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_var] = str(N_THREADS)

# Код решения
REPO_URL = "https://github.com/mishin-mikhail/avito_autumn_dev.git"
REPO_REF = "main"            


def _github_token():
    """GITHUB_TOKEN из окружения или из Kaggle Secrets (None, если его нет)."""
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


def _git(*args, token=None) -> str:
    """git без утечки токена: заголовок авторизации передаётся через переменные окружения."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        error = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"git завершился с ошибкой:\n{error}")
    return result.stdout.strip()


def find_repo_root() -> Path:
    """REPO_DIR → папки выше текущей → (Kaggle или GITHUB_TOKEN) клонирование в текущую папку."""
    candidates = [Path(os.environ["REPO_DIR"]).expanduser()] if os.environ.get("REPO_DIR") else []
    candidates += [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / "src" / "pipeline.py").exists():
            return path.resolve()
    token = _github_token()
    if not (token or Path("/kaggle/input").exists()):
        raise RuntimeError(
            "Не найден код решения (папка src/). Откройте ноутбук из папки notebooks/ клонированного "
            "репозитория или укажите путь к репозиторию в настройке REPO_DIR.")
    target = Path("/kaggle/working/avito-candgen") if Path("/kaggle/working").exists() else Path.cwd() / "avito-candgen"
    if not target.exists():
        _git("clone", "--quiet", REPO_URL, str(target), token=token)
    _git("-C", str(target), "fetch", "--quiet", "--tags", "--force", "origin", token=token)
    is_branch = subprocess.run(["git", "-C", str(target), "rev-parse", "--verify", "--quiet",
                                f"origin/{REPO_REF}"], capture_output=True).returncode == 0
    _git("-C", str(target), "checkout", "--quiet", "--force", "--detach",
         f"origin/{REPO_REF}" if is_branch else REPO_REF)
    return target


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
try:
    COMMIT = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
                            capture_output=True, text=True).stdout.strip() or "(не git-репозиторий)"
except FileNotFoundError:
    COMMIT = "(git не установлен)"


def ensure_packages(requirements: dict) -> None:
    """requirements: модуль → pip-спецификации. Ставит только то, чего нет."""
    missing = []
    for module, specs in requirements.items():
        try:
            importlib.import_module(module)
        except ImportError:
            missing += specs
    if not missing:
        return
    print("устанавливаю:", " ".join(missing))
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    if subprocess.run(cmd).returncode != 0 and subprocess.run(cmd + ["--user"]).returncode != 0:
        raise RuntimeError(f"Не удалось установить {missing}. Установите их вручную в терминале.")
    importlib.invalidate_caches()
    import site
    if site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())


ensure_packages({
    "pyarrow": ["pyarrow"],
    "pymorphy3": ["pymorphy3==2.0.6", "pymorphy3-dicts-ru==2.4.417150.4580142"],
    "transformers": ["transformers==4.57.6"],
})
try:
    import torch
except ImportError as error:
    raise RuntimeError("В окружении нет PyTorch. Ставить его нужно под версию CUDA этой машины - "
                       "команда есть на https://pytorch.org/get-started/locally/") from error
print(f"репозиторий: {REPO_ROOT}\nкоммит: {COMMIT}")

репозиторий: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev
коммит: 2adc8eb6767242624c852180d0c448bbeadcae79


In [ ]:
import json
import time
from dataclasses import replace

import numpy as np
import pandas as pd
from IPython.display import display

from src import encoder as enc
from src.config import CFG, EMB_CFG, EMB_CFG_LARGE, EMB_CFG_V6, RANKER_CFG
from src.data import load_benchmark, load_train
from src.paths import get_data_dir, get_work_dir
from src.pipeline import add_lemma_keys
from src.repro import library_versions, seed_everything
from src.sampling import build_folds, build_validation, group_table, picked_rows_mask, scheme_keys
from src.text import Lemmatizer
from src.utils import resources_report, timer
from src.validation import add_query_segments, mark_seen

seed_everything(CFG.seed)
SMOKE = os.environ.get("SMOKE_TEST") == "1"
DRY = os.environ.get("DRY_RUN") == "1" or SMOKE
START = time.perf_counter()

DATA_DIR, WORK_DIR = get_data_dir(), get_work_dir()

EMB_BASE = {"v4": EMB_CFG, "v6": EMB_CFG_V6, "large": EMB_CFG_LARGE}[EMB_VERSION]
E = replace(EMB_BASE, candidates=tuple(MODEL_CANDIDATES or EMB_BASE.candidates),
            batch_size=BATCH_SIZE or EMB_BASE.batch_size,
            train_pairs=EMB_BASE.train_pairs if TRAIN_PAIRS is None else TRAIN_PAIRS)
if E.init_model and not MODEL_CANDIDATES:

    init_path = WORK_DIR / E.init_model
    if not (init_path / "config.json").is_file() and not SMOKE:
        raise FileNotFoundError(f"Нет модели предыдущего раунда: {init_path}. Сначала нужен артефакт v4 "
                                "(03_embeddings с EMB_VERSION = \"v4\").")
    E = replace(E, candidates=(str(init_path),))
R = replace(RANKER_CFG, version="v4", uniform_unseen_texts=True)   
if DRY:          
    E = replace(E, train_pairs=2_000, zero_shot_queries=200, encode_batch=64) 
    R = replace(R, n_val_queries=300, fold_queries=300)

EMB_DIR = WORK_DIR / E.artifact_name
if (EMB_DIR / "manifest.json").is_file() and not OVERWRITE and not SMOKE:
    
    raise FileExistsError(f"Артефакт {EMB_DIR} уже готов. Проверьте EMB_VERSION в настройках; "
                          "перезаписать его можно только явно: OVERWRITE = True.")
GPU = enc.device_info()
AMP = enc.choose_amp(E.amp_dtype, GPU)

print(f"версия артефакта: {E.version} | режим: {'SMOKE_TEST' if SMOKE else 'DRY_RUN' if DRY else 'полный прогон'}")
print(f"модели: {list(E.candidates)}")
print(f"данные: {DATA_DIR}\nартефакт: {EMB_DIR}")
print(f"устройство: {GPU['name']}" + (f", {GPU['memory_gb']} ГБ, вычисления в {AMP.name}" if GPU["device"] == "cuda" else ""))
resources_report(WORK_DIR, need_ram_gb=10, need_disk_gb=5)
print(library_versions())
if GPU["device"] != "cuda" and not DRY:
    raise RuntimeError("GPU не найден: полный прогон на CPU займёт сутки. Проверьте, что ядро запущено "
                       "в GPU-окружении (nvidia-smi в терминале), или включите DRY_RUN для проверки кода.")

версия артефакта: large | режим: полный прогон
модели: ['intfloat/multilingual-e5-large']
данные: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/data
артефакт: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/artifacts/embeddings_large
устройство: NVIDIA A100 80GB PCIe MIG 2g.20gb, 19.5 ГБ, вычисления в bf16
RAM: 13.9 ГБ свободно из 16.0 | диск в /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/artifacts: 35 ГБ свободно | ядер CPU: 48
{'python': '3.10.13', 'numpy': '1.26.2', 'pandas': '2.0.3', 'scipy': '1.11.4', 'sklearn': '1.3.2', 'pyarrow': '14.0.1', 'pymorphy3': '2.0.6', 'lightgbm': '4.6.0', 'torch': '2.1.1+cu118'}


## 1. Данные и выборки

Строим те же выборки валидации и фолдов, что и в `04`-`06`. Их запросы исключаются из обучения энкодера, иначе он запомнит ответы и метрики ранкера будут завышены.

In [ ]:
with timer("загрузка"):
    train = load_train(DATA_DIR, with_description=True)
    bench_q, bench_items = load_benchmark(DATA_DIR)

lem = Lemmatizer()
with timer("подготовка запросов"):
    add_lemma_keys(train, lem)
    add_lemma_keys(bench_q, lem)
    item_locations = pd.Index(pd.concat([train["item_location_id"], bench_items["item_location_id"]]).unique())
    add_query_segments(train, item_locations)
    add_query_segments(bench_q, item_locations)
    mark_seen(bench_q, train["norm_text"].unique())
    groups = group_table(train, bench_items["item_id"])

with timer("выборки v4"):
    VAL = build_validation(train, groups, bench_q, R)
    KEYS = scheme_keys(groups)
    FOLDS = {name: build_folds(train, groups, bench_q, R, VAL[name], KEYS[name]) for name in VAL}
    SAMPLES = [*VAL.values(), *(fold for folds in FOLDS.values() for fold in folds)]

EXCLUDED = picked_rows_mask(train, SAMPLES)
print(f"выборок: {len(SAMPLES)} ({sum(len(s.queries) for s in SAMPLES):,} запросов)")
print(f"строк train: {len(train):,}; исключено из обучения энкодера: {EXCLUDED.sum():,} ({EXCLUDED.mean():.3f})")

[загрузка] 17.0 c
[подготовка запросов] 3.8 c
[warn] валидация in_corpus: в некоторых ячейках не хватило запросов — 2274 из 2500
[warn] фолд 0: в некоторых ячейках не хватило запросов — 3220 из 4000
[warn] фолд 1: в некоторых ячейках не хватило запросов — 2867 из 4000
[warn] фолд 2: в некоторых ячейках не хватило запросов — 2548 из 4000
[warn] фолд 3: в некоторых ячейках не хватило запросов — 2088 из 4000
[выборки v4] 25.1 c
выборок: 10 (31,497 запросов)
строк train: 497,673; исключено из обучения энкодера: 233,484 (0.469)


In [ ]:
bench_ids = frozenset(bench_items["item_id"])
extra_ids = sorted(dict.fromkeys(i for s in SAMPLES for rel in s.truth for i in rel if i not in bench_ids))
extra_items = (train[train["item_id"].isin(pd.Index(extra_ids))]
               .drop_duplicates("item_id", keep="first")[bench_items.columns].reset_index(drop=True))
QUERY_TEXTS_04 = list(dict.fromkeys(enc.query_texts(bench_q) + [t for s in SAMPLES for t in enc.query_texts(s.queries)]))

corpus_rows = np.arange(len(bench_items))
if DRY:           
    corpus_rows = corpus_rows[:5000]
CORPUS_ITEMS = pd.concat([bench_items.iloc[corpus_rows], extra_items], ignore_index=True)
CORPUS_TEXTS = enc.item_texts(CORPUS_ITEMS, E.desc_chars)
corpus_row = {v: i for i, v in enumerate(CORPUS_ITEMS["item_id"])}

eval_rows = train[EXCLUDED & train["item_id"].isin(pd.Index(CORPUS_ITEMS["item_id"])).to_numpy()]
eval_rows = eval_rows.drop_duplicates("query_key").head(E.zero_shot_queries)
EVAL_QUERIES = enc.query_texts(eval_rows)
EVAL_POSITIVE = np.array([corpus_row[i] for i in eval_rows["item_id"]])

free = np.flatnonzero(~EXCLUDED)
if E.train_pairs and len(free) > E.train_pairs:
    free = np.sort(np.random.default_rng(CFG.seed).choice(free, E.train_pairs, replace=False))
pairs_df = train.iloc[free]

print(f"объявлений к кодированию: {len(CORPUS_TEXTS):,} (корпус {len(corpus_rows):,} + подмешиваемые {len(extra_items):,})")
print(f"запросов для 04: {len(QUERY_TEXTS_04):,} | оценочных пар: {len(EVAL_QUERIES):,} | пар для обучения: {len(pairs_df):,}")
print("пример запроса:   ", EVAL_QUERIES[0][:150])
print("пример объявления:", CORPUS_TEXTS[0][:150])

объявлений к кодированию: 208,905 (корпус 189,212 + подмешиваемые 19,693)
запросов для 04: 25,638 | оценочных пар: 2,000 | пар для обучения: 150,000
пример запроса:    query: наращивание ресницы
пример объявления: passage: Ремонт/выкуп компьют. и ноутбуков с выездом на дом | Компьютерная помощь Компьютеры | Выезд на дом в любое время, вплоть до 23:00

● Диагност


Для large берём 150 тыс. пар из 264 тыс. свободных. Кодируем корпус бенчмарка вместе с эталонами выборок (209 тыс. объявлений) и все запросы, которые понадобятся в `04`-`08`.

## 2. Исходная модель

Recall@100: доля отложенных запросов, у которых выбранное объявление попало в топ-100 по близости векторов. Эти запросы энкодер не видел.

In [6]:
def evaluate(model, tag: str):
    """Recall@100 поиска по векторам на отложенных парах. Возвращает (вектора корпуса, recall)."""
    with timer(f"{tag}: кодирование корпуса"):
        item_emb = model.encode(CORPUS_TEXTS, E.max_len_item, E.encode_batch, log_every=50)
    q_emb = model.encode(EVAL_QUERIES, E.max_len_query, E.encode_batch)
    recall = enc.dense_recall(q_emb, item_emb, EVAL_POSITIVE, k=100, device=GPU["device"])
    print(f"{tag}: Recall@100 = {recall:.4f}")
    return item_emb, recall


ZERO_SHOT, SOURCES = {}, {}
if SMOKE:
    MODEL_NAME = "debug-random"
    model = enc.build_debug_encoder(CORPUS_TEXTS + EVAL_QUERIES, GPU["device"])
    ZERO_SHOT[MODEL_NAME] = evaluate(model, MODEL_NAME)[1]
else:
    for name in E.candidates:
        with timer(f"{name}: скачивание"):
            SOURCES[name] = enc.fetch_model(name)
    best = None
    for name in E.candidates:
        candidate = enc.BiEncoder.from_pretrained(SOURCES[name], GPU["device"], AMP)
        _, ZERO_SHOT[name] = evaluate(candidate, name)
        if best is None or ZERO_SHOT[name] > ZERO_SHOT[best]:
            best, model = name, candidate
        else:
            del candidate
    MODEL_NAME = best
    if GPU["device"] == "cuda":
        torch.cuda.empty_cache()
print(f"\n→ дообучаем: {MODEL_NAME}")

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

[intfloat/multilingual-e5-large: скачивание] 50.9 c
  закодировано 512 из 208,905
  закодировано 26,112 из 208,905
  закодировано 51,712 из 208,905
  закодировано 77,312 из 208,905
  закодировано 102,912 из 208,905
  закодировано 128,512 из 208,905
  закодировано 154,112 из 208,905
  закодировано 179,712 из 208,905
  закодировано 205,312 из 208,905
[intfloat/multilingual-e5-large: кодирование корпуса] 765.3 c
intfloat/multilingual-e5-large: Recall@100 = 0.3925

→ дообучаем: intfloat/multilingual-e5-large


## 3. Дообучение

InfoNCE: запрос притягивается к своему объявлению и отталкивается от объявлений остальных запросов батча и от двух трудных негативов. Чем больше батч, тем больше негативов, поэтому батч подбирается под память GPU пробными шагами.

Трудные негативы: похожие объявления корпуса, которые пользователь не выбрал. Самый верх выдачи пропускаем, а объявления той же микрокатегории исключаем: чаще всего они тоже подходят запросу, и отталкивать их значит учить модель ошибаться.

In [7]:
if E.batch_size == 0:
    with timer("подбор батча"):
        E = replace(E, batch_size=enc.probe_batch_size(model, E, candidates=tuple(
            b for b in (16, 32, 48, 64, 96, 128, 160, 192, 256, 320, 384, 512) if b <= E.max_batch_size)))
steps = len(pairs_df) // E.batch_size * E.epochs
print(f"батч: {E.batch_size} запросов → {E.batch_size * (2 + E.hard_negatives)} текстов на шаг, "
      f"негативов на запрос: {E.batch_size * (1 + E.hard_negatives) - 1}; шагов обучения: {steps:,}")

  доступно памяти GPU: 19.3 из 19.5 ГБ, используем до 16.4 ГБ
  батч 16: пик памяти 6.2 ГБ
  батч 32: пик памяти 7.2 ГБ
  батч 48: пик памяти 8.5 ГБ
  батч 64: пик памяти 9.8 ГБ
  батч 96: пик памяти 12.6 ГБ
  батч 128: пик памяти 15.3 ГБ
  батч 160: по прогнозу 18.0 ГБ — не проверяем
[подбор батча] 21.3 c
батч: 128 запросов → 512 текстов на шаг, негативов на запрос: 383; шагов обучения: 1,171


In [ ]:
TRAIN_QUERIES = enc.query_texts(pairs_df)
TRAIN_ITEM_TEXTS = enc.item_texts(pairs_df, E.desc_chars)

with timer("кодирование для поиска негативов"):
    corpus_emb = model.encode(CORPUS_TEXTS, E.max_len_item, E.encode_batch, log_every=100)
    train_q_emb = model.encode(TRAIN_QUERIES, E.max_len_query, E.encode_batch, log_every=100)

positive_in_corpus = np.array([corpus_row.get(i, -1) for i in pairs_df["item_id"]])

groups_kw = {}
if E.neg_exclude_same_micro:
    micro_codes, _ = pd.factorize(pd.concat([pairs_df["item_microcat_id"], CORPUS_ITEMS["item_microcat_id"]])
                                  .astype(str).to_numpy(), sort=True)
    groups_kw = dict(positive_group=micro_codes[:len(pairs_df)], item_group=micro_codes[len(pairs_df):])
with timer("трудные негативы"):
    neg_idx = enc.mine_hard_negatives(train_q_emb, corpus_emb, positive_in_corpus, E.hard_negatives,
                                      E.hard_neg_skip, E.hard_neg_depth, CFG.seed, GPU["device"], **groups_kw)
negatives = [[CORPUS_TEXTS[j] for j in row] for row in neg_idx]
pairs = enc.TrainPairs(queries=TRAIN_QUERIES, positives=TRAIN_ITEM_TEXTS, negatives=negatives)
del corpus_emb, train_q_emb
print("запрос:  ", TRAIN_QUERIES[0][:120])
print("позитив: ", TRAIN_ITEM_TEXTS[0][:120])
print("негатив: ", negatives[0][0][:120])

  закодировано 512 из 208,905
  закодировано 51,712 из 208,905
  закодировано 102,912 из 208,905
  закодировано 154,112 из 208,905
  закодировано 205,312 из 208,905
  закодировано 512 из 150,000
  закодировано 51,712 из 150,000
  закодировано 102,912 из 150,000
[кодирование для поиска негативов] 847.9 c
[трудные негативы] 18.2 c
запрос:   query: скупка телевизоров
позитив:  passage: Скупка б/у техники |  | Скупаю практически любую современную новую и б/у технику(обязательно рабочую), до 80% о
негатив:  passage: Скупка исправных и неисправных телевизоров | Ремонт и обслуживание техники Телевизоры | Выгодно покупаю-скупаю-


Без дообучения large даёт 0,3925 (e5-base: 0,384). Батч 128: следующий, 160, по прогнозу не помещается в 16,4 ГБ. Итого 1 171 шаг, по 383 негатива на запрос.

Фильтр по микрокатегории, как и в v6, убирает не все подходящие объявления: для «скупки телевизоров» негативом стала «Скупка исправных и неисправных телевизоров» из другой микрокатегории.

In [ ]:
with timer("дообучение"):
    model, history = enc.train_biencoder(model, pairs, E, CFG.seed, log_every=10 if DRY else 100)
model.save(EMB_DIR / "model")          
history.tail(5)

  шаг 1/1171: loss 3.3377
  шаг 100/1171: loss 1.2955
  шаг 200/1171: loss 1.2516
  шаг 300/1171: loss 1.2558
  шаг 400/1171: loss 1.1609
  шаг 500/1171: loss 1.0226
  шаг 600/1171: loss 1.1673
  шаг 700/1171: loss 1.0113


## 4. Результат

Во время обучения пропадала связь с удалённой средой: вывод обучения обрывается на шаге 700, а у ячеек ниже его нет. Обучение и запись артефакта завершились, числа ниже взяты из `manifest.json` и `history.json` артефакта.

Loss: 3,34 на первом шаге, 1,30 к сотому, 1,0-1,2 на шагах 500-1100.

In [ ]:
ITEM_EMB, RECALL_TUNED = evaluate(model, f"{MODEL_NAME} (дообученная)")
result = pd.Series({**ZERO_SHOT, f"{MODEL_NAME} (дообученная)": RECALL_TUNED}, name="Recall@100")
display(result.round(4).to_frame())

Recall@100 после дообучения **0,422** (без дообучения 0,3925, у e5-base v6 0,420). Одна large немного лучше по Recall@100, но в ранкере уступает v6; пользу она даёт в смеси с v6 (`08_ranker_v8.ipynb`).

## 5. Артефакт

Модель, вектора объявлений и запросов (float16) и манифест с параметрами, метриками и md5 файлов. `08_ranker_v8.ipynb` смешивает эти вектора с векторами v6.

In [ ]:
with timer("кодирование запросов для 04"):
    QUERY_EMB = model.encode(QUERY_TEXTS_04, E.max_len_query, E.encode_batch, log_every=50)

MANIFEST = enc.save_artifact(
    EMB_DIR, CORPUS_ITEMS["item_id"], ITEM_EMB, QUERY_TEXTS_04, QUERY_EMB,
    meta={"model_name": MODEL_NAME, "model_source": str(SOURCES.get(MODEL_NAME, MODEL_NAME)),
          "recall@100_zero_shot": ZERO_SHOT, "recall@100_tuned": RECALL_TUNED,
          "mode": "smoke" if SMOKE else "dry_run" if DRY else "full",
          "device": GPU, "amp": AMP.name, "commit": COMMIT,
          "emb_config": E.as_dict(), "ranker_config_for_samples": R.as_dict(),
          "seed": CFG.seed, "versions": library_versions()})
(EMB_DIR / "history.json").write_text(history.to_json(orient="records"))

print(json.dumps({k: MANIFEST[k] for k in ("model_name", "mode", "n_items", "n_queries", "dim", "md5")},
                 ensure_ascii=False, indent=1))
size_gb = sum(f.stat().st_size for f in EMB_DIR.rglob("*") if f.is_file()) / 2 ** 30
print(f"\nартефакт: {EMB_DIR} ({size_gb:.2f} ГБ) | всего времени: {(time.perf_counter() - START) / 60:.0f} мин")